# Лабораторная работа 3
## Основная фаза симплекс-метода

Реализация основной фазы симплекс-метода для решения задачи линейного программирования в канонической форме.


In [4]:
import numpy as np

def simplex_main_phase(c, A, x_init, B_init, max_iter=1000):
    """
    Основная фаза симплекс-метода.
    
    Параметры:
    c : np.ndarray
        Вектор коэффициентов целевой функции (на максимум).
    A : np.ndarray
        Матрица ограничений-равенств (m x n).
    x_init : np.ndarray
        Начальный базисный допустимый план (размер n).
    B_init : list или np.ndarray
        Набор индексов базисных переменных (размер m). Ожидается 0-индексация.
    max_iter : int
        Максимальное количество итераций (для защиты от зацикливания).
        
    Возвращает:
    status : str
        Сообщение о результате ('optimal' или 'unbounded').
    x : np.ndarray или None
        Оптимальный план, если целевая функция ограничена.
    B : list
        Индексы базисных переменных оптимального плана.
    """
    x = np.array(x_init, dtype=float)
    B = list(B_init)
    
    for iteration in range(max_iter):
        # 1. Построить базисную матрицу A_B и найти обратную
        A_B = A[:, B]
        try:
            A_B_inv = np.linalg.inv(A_B)
        except np.linalg.LinAlgError:
            raise ValueError("Матрица A_B вырождена. B не является базисом.")
            
        # 2. Сформировать вектор c_B
        c_B = c[B]
        
        # 3. Найти вектор потенциалов u
        u = c_B @ A_B_inv
        
        # 4. Найти вектор оценок delta
        delta = c - u @ A
        
        # 5. Проверка оптимальности (delta <= 0)
        # Учитываем вычислительную погрешность
        eps = 1e-9
        if np.all(delta <= eps):
            return "optimal", x, B
            
        # 6. Найти индекс первой положительной компоненты (j0)
        j0 = -1
        for j in range(len(delta)):
            if delta[j] > eps:
                j0 = j
                break
                
        # 7. Вычислить вектор z
        z = A_B_inv @ A[:, j0]
        
        # 8-9. Построить вектор theta и найти минимум
        theta = np.full(len(B), np.inf)
        for i in range(len(B)):
            if z[i] > eps:
                theta[i] = x[B[i]] / z[i]
                
        theta0 = np.min(theta)
        
        # 10. Проверка на неограниченность
        if np.isinf(theta0):
            return "unbounded", None, B
            
        # 11. Найти индекс k, на котором достигается минимум.
        k = np.argmin(theta)
        
        # 12-13. Замена базиса и обновление плана
        j_star = B[k]
        
        # Обновляем компоненты
        x[j0] = theta0
        for i in range(len(B)):
            if i != k:
                x[B[i]] = x[B[i]] - theta0 * z[i]
        x[j_star] = 0
        
        B[k] = j0
        
    raise RuntimeError("Превышено максимальное число итераций!")


In [5]:
# Пример 1: Задача имеет оптимальный план
c1 = np.array([1.0, 1.0, 0.0, 0.0, 0.0])
A1 = np.array([
    [-1.0,  1.0, 1.0, 0.0, 0.0],
    [ 1.0,  0.0, 0.0, 1.0, 0.0],
    [ 0.0,  1.0, 0.0, 0.0, 1.0]
])
x_init1 = np.array([0.0, 0.0, 1.0, 3.0, 2.0])
B_init1 = [2, 3, 4]

status1, x_opt1, B_opt1 = simplex_main_phase(c1, A1, x_init1, B_init1)

print("--- Пример 1 ---")
print("Статус:", status1)
if status1 == 'optimal':
    print("Оптимальный план:", np.round(x_opt1, 5))
    print("Индексы базиса (0-based):", B_opt1)
    print("Значение целевой функции:", np.round(np.dot(c1, x_opt1), 5))

# Пример 2: Целевая функция неограничена
c2 = np.array([1.0, 0.0, 0.0, 0.0])
A2 = np.array([
    [ 1.0, -1.0, 1.0, 0.0],
    [-1.0,  1.0, 0.0, 1.0]
])
x_init2 = np.array([1.0, 0.0, 0.0, 3.0])
B_init2 = [0, 3]

status2, x_opt2, B_opt2 = simplex_main_phase(c2, A2, x_init2, B_init2)

print("\n--- Пример 2 ---")
print("Статус:", status2)
if status2 == 'optimal':
    print("Оптимальный план:", x_opt2)
else:
    print("Целевой функционал неограничен сверху на множестве допустимых планов.")

--- Пример 1 ---
Статус: optimal
Оптимальный план: [3. 2. 2. 0. 0.]
Индексы базиса (0-based): [2, 0, 1]
Значение целевой функции: 5.0

--- Пример 2 ---
Статус: unbounded
Целевой функционал неограничен сверху на множестве допустимых планов.
